# Proyecto final - Machine Learning #
#### Grupo 2 : Ineta Keryte, Anthonny Maldonado, Guillermo Mansanta ####

Este es el proyecto final de nuestro bootcamp de Machine Learning, donde demostramos las habilidades y conocimientos adquiridos a lo largo de nuestros estudios. A lo largo de este bootcamp, hemos estudiado diferentes modelos basados en proyectos de diferentes áreas y tipos. Ahora es el momento de crear nuestro propio proyecto utilizando el algoritmo que creemos que se adapta mejor a nuestro problema.

In [94]:
import pandas as pd
import sqlite3
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
import streamlit as st
import pickle
import re
import json

### Nuestro problema a resolver ###

<p style="text-align: justify;">
En el contexto macroeconómico argentino, atravesado por inflación, inestabilidad en los precios y restricciones en el acceso al financiamiento, la gestión de inventarios se convierte en un factor crítico para la sostenibilidad de cualquier farmacia. La falta de una planificación de stock basada en criterios técnicos y analíticos impacta directamente tanto en la rentabilidad del negocio como en la calidad del servicio prestado a la comunidad.

Desde la perspectiva comercial, una mala política de inventarios genera una utilización ineficiente del capital de trabajo, con recursos financieros inmovilizados en mercadería de baja rotación o con riesgo de vencimiento. Esto incrementa los costos operativos, deteriora el flujo de caja y limita la capacidad de negociación con droguerías y laboratorios, afectando condiciones de pago, descuentos y líneas de crédito.

Desde la perspectiva del servicio farmacéutico, los errores de planificación derivan en quiebres de stock de medicamentos esenciales, demoras en la atención, pérdida de continuidad en tratamientos y disminución de la confianza de los pacientes y clientes. La farmacia deja de ser percibida como un punto de referencia sanitario confiable y pasa a ser vista como un comercio reactivo e ineficiente.

En conjunto, la ausencia de una gestión profesional del inventario compromete simultáneamente la competitividad económica del negocio y su rol social como prestador de un servicio de salud.

👉 ## Problema real: la farmacia no cuenta con una metodología objetiva para anticipar la demanda futura de sus productos.
</p>


### Objetivo de nuestro proyecto ### 

<p style="text-align: justify;">
Este proyecto tiene como objetivo aplicar técnicas de Machine Learning para mejorar la forma en que una farmacia gestiona su stock. A partir del análisis de las ventas de un año completo (2025) de una farmacia ubicada en la provincia de Buenos Aires, se busca entender cómo se comporta la demanda de los productos y usar esa información para planificar mejor el inventario del año 2026. La idea principal es pasar de una gestión basada solo en la experiencia a una gestión basada en datos, que permita anticiparse a las necesidades de los clientes, evitar faltantes de productos importantes y reducir el exceso de mercadería en un contexto económico cambiante. Todo el enfoque está pensado desde la realidad del negocio farmacéutico y el comportamiento de consumo de las personas.
</p>

### Dataset - Carga de conjunto de datos

In [95]:
df = pd.read_csv("../data/raw/farmacia-datos.csv", sep=";", encoding="latin1")
df.head()

,Fecha,Tipo Mov.,Fac. Tipo,Fac. Suc.,Fac. Nun.,Fisc. Numero,Tipo Pago,Cant.,Precio,Producto,Sub. Total,Rubro,Cobertura,Ajustes,Desc. Adic.,Total. Cliente,IVA,Tasa Iva,Total Gravado,Total sin Gravar
0,01/01/25 01.13.46,F,B,0,379923,NaN,E,1,9344,ACTRON PEDIATRICO 4% susp.oral x 100 ml,9344,FARMACIA,0,0,0,9344,0,0,0,9344
1,01/01/25 01.59.21,F,B,0,379924,NaN,E,1,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,"8174,96",FARMACIA,"3032,85",0,0,"5142,11",0,0,0,"8174,96"
2,01/01/25 02.01.59,F,B,0,379925,NaN,E,1,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,"8174,96",FARMACIA,"5318,1",0,0,"2856,86",0,0,0,"8174,96"
3,01/01/25 02.04.14,F,B,0,379926,NaN,E,1,"8086,82",MUELITA FORTE GEL gel pomo x 10 g,"8086,82",FARMACIA,0,0,0,"8086,82",0,0,0,"8086,82"
4,01/01/25 02.08.35,F,B,0,379927,NaN,E,1,"26750,12",DIOXAFLEX B12 comp.x 20,"26750,12",FARMACIA,0,0,0,"26750,12",0,0,0,"26750,12"


<p style="text-align: justify;">
El dataset contiene información de registro de ventas durante el año 2025 de una farmacia situada en Argentina, en la provincia de Buenos Aires. El registro se corresponde a los datos de los tickets de venta generados durante todo el año, los días que el comercio estuvo abierto. Teniendo en cuenta que el comercio trabaja de Lunes a Sábados de 8hs a 20 hs, es decir, 12 hs por día, se han generado un total de datos tal que nuestro dataset contiene:
</p>

- 117.415 filas

- 20 columnas, con variables categoricas y numericas como: ['Fecha', 'Tipo Mov.', 'Fac. Tipo', 'Fac. Suc.', 'Fac. Nun.','Fisc. Numero', 'Tipo Pago', 'Cant.', 'Precio', 'Producto', 'Sub. Total', 'Rubro', 'Cobertura', 'Ajustes', 'Desc. Adic.', 'Total. Cliente', 'IVA', 'Tasa Iva', 'Total Gravado', 'Total sin Gravar']

- Más de 7.500 productos distintos

- Rubro farmacia (medicamentos, insumos médicos, suplementos, vitaminas, salud preventiva) y perfumería y cuidado personal (cremas, protectores, higiene).
<p style="text-align: justify;">
Se trata de un conjunto de datos reales, lo que implica la presencia de ruido, valores inconsistentes, formatos heterogéneos y registros incompletos, características habituales en fuentes operativas del sector farmacéutico. Esta naturaleza del dataset representó un desafío significativo durante la etapa de data cleaning, ya que fue necesario aplicar múltiples técnicas de depuración, normalización y validación para garantizar la calidad de los datos antes de avanzar con el análisis y el modelado.
</p>

In [96]:
df.shape

(117415, 20)

In [97]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117415 entries, 0 to 117414
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Fecha             117415 non-null  object 
 1   Tipo Mov.         117415 non-null  object 
 2   Fac. Tipo         117415 non-null  object 
 3   Fac. Suc.         117415 non-null  int64  
 4   Fac. Nun.         117415 non-null  int64  
 5   Fisc. Numero      0 non-null       float64
 6   Tipo Pago         117415 non-null  object 
 7   Cant.             117415 non-null  int64  
 8   Precio            117415 non-null  object 
 9   Producto          117414 non-null  object 
 10  Sub. Total        117415 non-null  object 
 11  Rubro             117415 non-null  object 
 12  Cobertura         117415 non-null  object 
 13  Ajustes           117415 non-null  int64  
 14  Desc. Adic.       117415 non-null  object 
 15  Total. Cliente    117415 non-null  object 
 16  IVA               11

In [98]:
df.duplicated().sum()

np.int64(121)

In [99]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
df.duplicated().sum()

np.int64(0)

### Observaciones

 - Se logran visualizar los datos, en cada fila podemos darnos cuenta que es una venta. Alguna de ellas comparten el mismo numero de ticket, lo cual no da a entender que hay varios articulos dentro de un mismo ticket.
 - Encontramos unos duplicados pero para descartarlas tomamos la decision de evaluarlas antes para tener una decision mas acertada. Tenemos 20 columnas y estudiaremos cada una de ellas.
 - Prodecimos a eliminar la columna `Fisc. Numero` dado que no tiene ningun valor en ninguna fila.

In [100]:
df.drop(["Fisc. Numero"],axis=1, inplace=True)

### Almacenamiento de información

In [101]:
df.columns = df.columns.str.replace('.', '', regex=False).str.replace(' ', '_')

Para la ejecución de las consultas SQL se normalizaron los nombres de columnas para garantizar la correcta agregación de variables numéricas.

In [102]:
# Create connection to SQLite
conn = sqlite3.connect('farmacia-datos.db')

In [103]:
# Save table
df.to_sql('ventas', conn, if_exists='replace', index=False)

117294

In [104]:
# Verify that the table exists
query = 'SELECT * FROM ventas LIMIT 5;'
pd.read_sql(query, conn)

,Fecha,Tipo_Mov,Fac_Tipo,Fac_Suc,Fac_Nun,Tipo_Pago,Cant,Precio,Producto,Sub_Total,Rubro,Cobertura,Ajustes,Desc_Adic,Total_Cliente,IVA,Tasa_Iva,Total_Gravado,Total_sin_Gravar
0,01/01/25 01.13.46,F,B,0,379923,E,1,9344,ACTRON PEDIATRICO 4% susp.oral x 100 ml,9344,FARMACIA,0,0,0,9344,0,0,0,9344
1,01/01/25 01.59.21,F,B,0,379924,E,1,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,"8174,96",FARMACIA,"3032,85",0,0,"5142,11",0,0,0,"8174,96"
2,01/01/25 02.01.59,F,B,0,379925,E,1,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,"8174,96",FARMACIA,"5318,1",0,0,"2856,86",0,0,0,"8174,96"
3,01/01/25 02.04.14,F,B,0,379926,E,1,"8086,82",MUELITA FORTE GEL gel pomo x 10 g,"8086,82",FARMACIA,0,0,0,"8086,82",0,0,0,"8086,82"
4,01/01/25 02.08.35,F,B,0,379927,E,1,"26750,12",DIOXAFLEX B12 comp.x 20,"26750,12",FARMACIA,0,0,0,"26750,12",0,0,0,"26750,12"


### Top 10 productos mas vendidos.

In [105]:
# Execute a real SQL query
query_top_10 = '''
SELECT Producto, SUM("Cant") AS Demanda_total
FROM ventas
GROUP BY Producto
ORDER BY Demanda_total DESC
LIMIT 10;
'''
pd.read_sql(query_top_10, conn)

,Producto,Demanda_total
0,BUSCAPINA COMPOSITUM comp.rec.x 20,2648
1,QURA PLUS comp.rec.x 20,2601
2,GEN LP IBUPROFENO blister 600 mg x 10,2134
3,SERTAL COMPUESTO comp.rec.x 20,1786
4,GEN LP OMEPRAZOL blister 20 mg x 15,1574
5,ALIKAL (unica),1483
6,MYLANTA EXTRA comp.mast.x 24,1270
7,ASPIRINA PREVENT comp.cub.enterica x 50,1159
8,CAFIASPIRINA comp.x 30,1040
9,TAFIROL 1G X 8 (unica),983


> - Se almacenaron los datos en una base de datos SQLite y se realizaron consultas SQL desde Python para identificar productos de alta rotación y patrones de demanda.

#### Demanda total por producto

In [106]:
query_product = '''
SELECT Producto, SUM("Cant") AS Demanda_total
FROM ventas
GROUP BY Producto;
'''
pd.read_sql(query_product, conn)

,Producto,Demanda_total
0,None,1
1,CHUP.MANZANITA(C/DIBUJO)T/SILIC.+3M RED - ES...,1
2,(unica),4
3,1,1
4,1 1,7
...,...,...
7671,q,1
7672,|,1
7673,º,3
7674,ÓLEO Calcáreo x 240 ml.,1


> - Esta consulta permite identificar los productos con mayor volumen de ventas acumuladas, fundamentales para la gestión de stock y la priorización de reposición.

#### Demanda sumada en dias

In [107]:
query_per_day = """
SELECT strftime('%Y-%m', Fecha) AS mes, SUM("Cant") AS Demanda_mensual
FROM ventas
GROUP BY mes
ORDER BY mes;
"""

pd.read_sql(query_per_day, conn)

,mes,Demanda_mensual
0,None,160495


> - La suma diaria por producto permite modelar la demanda como una serie temporal, base para la predicción de consumo futuro.

In [108]:
query_coverage = """
SELECT Cobertura, SUM("Cant") AS Demanda_total
FROM ventas
GROUP BY Cobertura;
"""
pd.read_sql(query_coverage, conn)


,Cobertura,Demanda_total
0,"-157953,92",1
1,0,107039
2,10000,1
3,"10000,188",1
4,"10001,04",3
...,...,...
23016,"9993,308",1
23017,"9995,3",3
23018,"9998,392",2
23019,"9998,952",1


> - La presencia de cobertura médica incrementa la demanda, lo cual debe considerarse en la planificación de inventario.

## Análisis descriptivo de las variables

Variable clave: `Cant`

In [109]:
df['Cant'].describe()

count    117294.000000
mean          1.368314
std           1.862667
min           0.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         106.000000
Name: Cant, dtype: float64

In [110]:
average = df['Cant'].mean()
median = df['Cant'].median()
mode = df['Cant'].mode()[0]
variance = df['Cant'].var()
skewness = df['Cant'].skew()

In [111]:
print(f'Media:     {average:.2f}')
print(f'Mediana:   {median:.2f}')
print(f'Moda:      {mode:.2f}')
print(f'Varianza:  {variance:.2f}')
print(f'Asimetría: {skewness:.2f}')

Media:     1.37
Mediana:   1.00
Moda:      1.00
Varianza:  3.47
Asimetría: 12.14


- **Media** (1.37) vs **Mediana** (1.00): El hecho de que la media sea mayor que la mediana indica que la distribución está sesgada a la derecha. Mientras que la mayoría de los clientes compran solo 1 unidad (mediana), hay un grupo pequeño de transacciones con cantidades muy altas que elevan el promedio.
- **Moda** (1.00): Es el valor más frecuente. Esto confirma que el comportamiento estándar en la farmacia es la compra de una única unidad por producto (típico de medicamentos bajo receta).
- **Varianza** (3.47): La desviación estándar sería la raíz cuadrada de esto (≈1.86). Aunque la mayoría compra 1 unidad, hay una variabilidad considerable. En términos de negocio, esto sugiere que conviven dos tipos de clientes: el consumidor final (compra 1 unidad) y posiblemente clientes institucionales o crónicos (compran varias cajas o tratamientos completos).
- **Asimetría Positiva Extrema**: Un valor de 12.14 es extremadamente alto (en una distribución normal sería 0).   

Debido a esta asimetría, será necesario decidir si se eliminan esos valores extremos para entrenar el modelo de Machine Learning o si se tratan de forma especial, ya que podrían distorsionar las predicciones.

In [112]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Fac_Suc,117294.0,0.000000,0.000000,0.0,0.00,0.0,0.0,0.0
Fac_Nun,117294.0,417631.799342,22418.434407,379923.0,398025.25,416261.0,437601.0,455865.0
Cant,117294.0,1.368314,1.862667,0.0,1.00,1.0,1.0,106.0
Ajustes,117294.0,0.000000,0.000000,0.0,0.00,0.0,0.0,0.0
Tasa_Iva,117294.0,4.085631,8.313029,0.0,0.00,0.0,0.0,21.0


In [113]:
df['Fecha'] = pd.to_datetime(df['Fecha'], format='%d/%m/%y %H.%M.%S')

### Limpieza de datos: eliminación de información que detectamos que hace más ruido de lo que nos sirve.

In [114]:
products_no_name = df[df['Producto'].str.len() <= 4]['Producto'].unique()
products_no_name

array(['1 1', 'A1 1', '5200', 'º', '|', 'LIMA', 'q', '1'], dtype=object)

In [115]:
df = df[df['Producto'].str.len() > 4]

> - Se eliminaron registros cuyos nombres de producto tenían una longitud menor o igual a cuatro caracteres, ya que correspondían a valores erróneos o incompletos que no aportaban información relevante al análisis.

In [116]:
(df['Fac_Suc'] == 0).sum()

np.int64(117277)

In [117]:
df['Tipo_Mov'].unique()

array(['F'], dtype=object)

In [118]:
df['Fac_Tipo'].value_counts(dropna=False)

Fac_Tipo
B    117114
A       163
Name: count, dtype: int64

In [119]:
df['IVA'].value_counts(dropna=False)

IVA
0              94465
173,553719       982
520,661157       673
867,768595       658
780,9917355      641
               ...  
1818,81            1
2965,42562         1
3529,132959        1
1662,644628        1
4252,447517        1
Name: count, Length: 4217, dtype: int64

In [120]:
df['Tasa_Iva'].value_counts(dropna=False)

Tasa_Iva
0     94465
21    22812
Name: count, dtype: int64

In [121]:
df['Tipo_Pago'].value_counts(dropna=False)

Tipo_Pago
E    117277
Name: count, dtype: int64

In [122]:
df['Ajustes'].value_counts(dropna=False)

Ajustes
0    117277
Name: count, dtype: int64

In [123]:
df['Desc_Adic'].value_counts(dropna=False)

Desc_Adic
0             115135
-3,64E-12        226
-1,82E-12         86
-7,28E-12         79
-9,09E-13         24
               ...  
1450               1
8741,37096         1
4704,488           1
231,56             1
1148               1
Name: count, Length: 1155, dtype: int64

In [124]:
df["Producto"].value_counts()

Producto
GEN LP IBUPROFENO blister 600 mg x 10      1925
GEN LP OMEPRAZOL blister 20 mg x 15        1203
ASPIRINA PREVENT comp.cub.enterica x 50    1149
TAFIROL 1G X 8 (unica)                      870
TAFIROL 1 G comp.ran.x 50                   699
                                           ... 
TOBILLERA (unica)                             1
MECANYL DUO cáps.x 60                         1
FLEVOMAX 1000 mg comp.rec.x 30                1
ACCESORIOS LUCIA (unica)                      1
LRP ANTHELIOS XL SPF 50 FLU CO                1
Name: count, Length: 7667, dtype: int64

In [125]:
df["Rubro"].value_counts()

Rubro
FARMACIA           108266
PERFUMERIA           8910
NO ESPECIFICADO       101
Name: count, dtype: int64

In [127]:
df["Total_Gravado"].value_counts()

Total_Gravado
0              94465
826,446281       982
2479,338843      673
4132,231405      658
3719,008264      641
               ...  
8661               1
14121,07438        1
16805,39504        1
7917,355372        1
20249,75008        1
Name: count, Length: 4217, dtype: int64

In [131]:
df["Total_sin_Gravar"].value_counts()

Total_sin_Gravar
0           21995
3500         1410
5000         1376
4000         1316
3000          758
            ...  
9783            1
6942,08         1
26376,02        1
19010           1
9646,28         1
Name: count, Length: 26927, dtype: int64

In [129]:
df.columns

Index(['Fecha', 'Tipo_Mov', 'Fac_Tipo', 'Fac_Suc', 'Fac_Nun', 'Tipo_Pago',
       'Cant', 'Precio', 'Producto', 'Sub_Total', 'Rubro', 'Cobertura',
       'Ajustes', 'Desc_Adic', 'Total_Cliente', 'IVA', 'Tasa_Iva',
       'Total_Gravado', 'Total_sin_Gravar'],
      dtype='object')

In [133]:
df.dtypes

Fecha               datetime64[ns]
Tipo_Mov                    object
Fac_Tipo                    object
Fac_Suc                      int64
Fac_Nun                      int64
Tipo_Pago                   object
Cant                         int64
Precio                      object
Producto                    object
Sub_Total                   object
Rubro                       object
Cobertura                   object
Ajustes                      int64
Desc_Adic                   object
Total_Cliente               object
IVA                         object
Tasa_Iva                     int64
Total_Gravado               object
Total_sin_Gravar            object
dtype: object

#### Procederemos a ejecutar las siguientes acciones.

>1. Borraremos las variables `Tipo Mov.` y `Tipo Pago` por que todas las variables de cada columna tienen un valor unico que se repite. no aportaria nada realmente al Dataset y procederemos a borrarlo.
>2. Las variables `Precio`, `Sub. Total`, `Cobertura`, `Desc. Adic.`, `Total. Cliente`, `IVA`, `Total Gravado`, `Total sin Gravar` notamos que son variables numericas. las cambiaremos a flotantes para poder trabajarlas como variables numericas. 

In [134]:
cat_variables = df.select_dtypes(include=['object']).columns.tolist()
cat_variables

['Tipo_Mov',
 'Fac_Tipo',
 'Tipo_Pago',
 'Precio',
 'Producto',
 'Sub_Total',
 'Rubro',
 'Cobertura',
 'Desc_Adic',
 'Total_Cliente',
 'IVA',
 'Total_Gravado',
 'Total_sin_Gravar']

In [135]:
df.drop(["Tipo_Mov", "Tipo_Pago"],axis=1, inplace=True)

In [136]:
columns_to_numeric = ['Precio', 'Sub_Total', 'Cobertura', 'Desc_Adic', 'Total_Cliente', 'IVA', 'Total_Gravado', 'Total_sin_Gravar']

for col in columns_to_numeric:
    df[col] = (df[col].astype(str)
                      .str.replace(',', '.')
                      .astype(float))